# Chatbot RAG — Ingesta del PDF (app_llm.ipynb)
**Curso: Tecnologías Emergentes · Semana 10**

Este notebook realiza la **ingesta**: lee el PDF, lo fragmenta en trozos de 1000 caracteres, genera los embeddings con `gemini-embedding-001` (3072 dimensiones), los guarda en MongoDB Atlas (`pdf_embeddings_db.pdf_vectors`) y crea el índice vectorial `vector_index`.

Al final incluye una sección de **pruebas** que genera la tabla de 5 preguntas y el análisis de `k = 3, 5, 10` para el informe.

| Paso | Sección |
|---|---|
| 2 | Librerías y secretos de Colab |
| 3 | Clientes de Gemini y MongoDB |
| 4 | Leer el PDF |
| 5 | Generar embeddings |
| 6 | Crear el índice vectorial |
| 7 | Fragmentar e insertar |
| — | Pruebas: 5 preguntas y análisis de k |

## Paso 2. Librerías y secretos
Registre `GOOGLE_API_KEY` y `MONGODB_URI` en el panel 🔑 **Secrets** de Colab y active *Notebook access*. Ninguna credencial se escribe en el notebook.

In [ ]:
!pip -q install pymongo google-genai PyPDF2 pandas

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
MONGODB_URI    = userdata.get('MONGODB_URI')
print("Secretos cargados ✔")

## Paso 3. Conectar con Gemini y MongoDB
La base `pdf_embeddings_db` y la colección `pdf_vectors` se crean automáticamente con la primera inserción.

> **Tarea extra:** para el segundo documento cambie `NOMBRE_COLECCION` (por ejemplo `reglamento_vectors`) y `PDF_PATH`.

In [ ]:
import time
import pymongo
from google import genai
from google.genai import types
from PyPDF2 import PdfReader

PDF_PATH         = "aws-cloud-adoption-framework_XL.pdf"   # se descarga abajo si no existe
NOMBRE_BD        = "pdf_embeddings_db"
NOMBRE_COLECCION = "pdf_vectors"
NOMBRE_INDICE    = "vector_index"
MODELO_EMBEDDING = "gemini-embedding-001"   # 3072 dimensiones
MODELOS_LLM      = ["gemini-3.1-flash-lite", "gemini-3.5-flash", "gemini-3.8-flash"]  # gemini-2.5-flash ya no está disponible para cuentas nuevas
TAMANO_CHUNK     = 1000

client_genai = genai.Client(api_key=GOOGLE_API_KEY, http_options=types.HttpOptions(timeout=40000))  # 40 s máx. por llamada
client_mongo = pymongo.MongoClient(MONGODB_URI)
db = client_mongo[NOMBRE_BD]
collection = db[NOMBRE_COLECCION]

client_mongo.admin.command("ping")
print("Conexión a MongoDB Atlas OK ✔")

Si el PDF de clase no está en el panel de archivos, se descarga el whitepaper oficial *Overview of the AWS Cloud Adoption Framework*.

In [ ]:
import os, urllib.request
URL_PDF = "https://docs.aws.amazon.com/pdfs/whitepapers/latest/overview-aws-cloud-adoption-framework/overview-aws-cloud-adoption-framework.pdf"
if not os.path.exists(PDF_PATH):
    urllib.request.urlretrieve(URL_PDF, PDF_PATH)
print(PDF_PATH, os.path.getsize(PDF_PATH), "bytes")

## Paso 4. Leer el PDF
`or ""` evita errores en páginas sin texto (por ejemplo, escaneadas).

In [ ]:
def leer_pdf(path_pdf):
    reader = PdfReader(path_pdf)
    texto = ""
    for page in reader.pages:
        texto += (page.extract_text() or "") + "\n"
    return texto.strip()

texto = leer_pdf(PDF_PATH)
print(f"Caracteres extraídos: {len(texto):,}")
print(texto[:500])

## Paso 5. Generar embeddings
En la ingesta se usa `task_type="RETRIEVAL_DOCUMENT"` porque se vectorizan fragmentos del documento. Se agrega un reintento por si se supera el límite de la capa gratuita (error 429).

In [ ]:
def crear_embedding(texto, task_type="RETRIEVAL_DOCUMENT"):
    for intento in range(5):
        try:
            response = client_genai.models.embed_content(
                model=MODELO_EMBEDDING,
                contents=texto,
                config=types.EmbedContentConfig(task_type=task_type))
            return response.embeddings[0].values
        except Exception as e:
            if any(c in str(e) for c in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")):
                print("Gemini saturado o límite gratuito alcanzado; esperando 20 s...")
                time.sleep(20)
            else:
                raise
    raise RuntimeError("No se pudo generar el embedding tras 5 intentos")

print("Dimensión del embedding:", len(crear_embedding("prueba")))

## Paso 6. Crear el índice vectorial
| Parámetro | Valor |
|---|---|
| name / type | `vector_index` / `vectorSearch` |
| path | `embedding` |
| similarity | `dotProduct` |
| numDimensions | 3072 |

In [ ]:
from pymongo.operations import SearchIndexModel

def crear_indice_vectorial():
    modelo = SearchIndexModel(
        definition={"fields": [{
            "type": "vector", "path": "embedding",
            "numDimensions": 3072, "similarity": "dotProduct"}]},
        name=NOMBRE_INDICE,
        type="vectorSearch")
    try:
        collection.create_search_index(model=modelo)
        print("Índice creado; esperando a que esté READY...")
    except Exception as e:
        print("Aviso (puede que ya exista):", e)

def esperar_indice():
    while True:
        info = list(collection.list_search_indexes(NOMBRE_INDICE))
        if info and info[0].get("queryable"):
            print("Índice READY ✔")
            return
        time.sleep(5)

## Paso 7. Fragmentar e insertar
Fragmentación de **tamaño fijo por caracteres** (1000, sin solapamiento). Se borran los documentos previos para que la ingesta pueda repetirse sin duplicar fragmentos.

In [ ]:
def procesar_pdf(texto):
    trozos = [texto[i:i+TAMANO_CHUNK] for i in range(0, len(texto), TAMANO_CHUNK)]
    documentos = []
    for i, chunk in enumerate(trozos):
        documentos.append({"id": i, "texto": chunk, "embedding": crear_embedding(chunk)})
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(trozos)} embeddings")
    collection.delete_many({})
    collection.insert_many(documentos)
    print(f"Fragmentos insertados: {len(documentos)}")

procesar_pdf(texto)          # con el PDF de clase deben ser ~84 fragmentos
crear_indice_vectorial()     # la colección ya existe, así que el índice se puede crear
esperar_indice()

# Atlas tarda unos segundos en indexar los vectores recién insertados: esperar a que la búsqueda devuelva resultados
vector_prueba = crear_embedding("prueba", "RETRIEVAL_QUERY")
for _ in range(24):
    n = len(list(collection.aggregate([{"$vectorSearch": {"index": NOMBRE_INDICE, "path": "embedding",
        "queryVector": vector_prueba, "numCandidates": 100, "limit": 5}}])))
    if n == 5:
        break
    time.sleep(5)
print("Búsqueda vectorial lista ✔ | Documentos en la colección:", collection.count_documents({}))

---
## Pruebas para el informe
Las mismas funciones que usa `app.py`: la pregunta se vectoriza con `RETRIEVAL_QUERY`, se buscan los k fragmentos con `$vectorSearch` y se genera la respuesta con Gemini 3.x (lista de respaldo, porque `gemini-2.5-flash` ya no está disponible para cuentas nuevas).

In [ ]:
import pandas as pd

def buscar_similares(pregunta, k=5):
    pipeline = [
        {"$vectorSearch": {
            "index": NOMBRE_INDICE, "path": "embedding",
            "queryVector": crear_embedding(pregunta, "RETRIEVAL_QUERY"),
            "numCandidates": max(100, k * 10), "limit": k}},
        {"$project": {"_id": 0, "id": 1, "texto": 1, "score": {"$meta": "vectorSearchScore"}}}]
    return list(collection.aggregate(pipeline))

def generar_respuesta(pregunta, fragmentos):
    contexto = "\n\n".join(f["texto"] for f in fragmentos)
    prompt = f'''Eres un asistente experto en adopción de la nube. Usa EXCLUSIVAMENTE el siguiente contexto para responder la pregunta del usuario. Si la respuesta no está en el contexto, indícalo claramente.

Contexto:
{contexto}

Pregunta: {pregunta}

Responde de forma concisa y clara en español.'''
    ultimo_error = None
    for ronda in range(3):              # hasta 3 rondas; pasa al siguiente modelo si uno está saturado (503)
        for modelo in MODELOS_LLM:
            try:
                return client_genai.models.generate_content(model=modelo, contents=prompt).text
            except Exception as e:
                ultimo_error = e
        time.sleep(5 * (ronda + 1))
    raise ultimo_error

def responder_seguro(pregunta, fragmentos):
    # Igual que generar_respuesta, pero si Gemini sigue saturado devuelve un aviso en vez de detener el notebook
    try:
        return generar_respuesta(pregunta, fragmentos).strip()
    except Exception as e:
        return f"(sin respuesta: {type(e).__name__}; vuelva a ejecutar la celda)"

### 5 preguntas (3 con respuesta en el documento y 2 sin ella)
Ajuste las preguntas si su PDF es otro.

In [ ]:
PREGUNTAS = [
    ("¿Cuáles son las seis perspectivas del AWS Cloud Adoption Framework?",          "Sí"),
    ("¿Qué capacidades incluye la perspectiva de seguridad?",                          "Sí"),
    ("¿Cuáles son las fases del recorrido de transformación (Envision, Align, Launch, Scale)?", "Sí"),
    ("¿Cuál es el precio por hora de una instancia EC2 t3.micro en São Paulo?",        "No"),
    ("¿Quién ganó la Copa Mundial de Fútbol de 2022?",                               "No"),
]

filas = []
for pregunta, en_doc in PREGUNTAS:
    frag = buscar_similares(pregunta, k=5)
    resp = responder_seguro(pregunta, frag)
    filas.append({
        "pregunta": pregunta, "¿en el doc.?": en_doc, "respuesta": resp,
        "ids recuperados": [f["id"] for f in frag],
        "scores": [round(f["score"], 3) for f in frag],
    })
    time.sleep(2)

tabla = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", 300)
tabla

### Análisis de k = 3, 5, 10
Para cada pregunta se registra el score máximo y mínimo del top k, la longitud del contexto enviado y la respuesta.

In [ ]:
filas_k = []
for pregunta, en_doc in PREGUNTAS:
    for k in (3, 5, 10):
        frag = buscar_similares(pregunta, k=k)
        resp = responder_seguro(pregunta, frag)
        filas_k.append({
            "pregunta": pregunta[:60], "¿en el doc.?": en_doc, "k": k,
            "score máx": round(frag[0]["score"], 3) if frag else None,
            "score mín": round(frag[-1]["score"], 3) if frag else None,
            "caracteres de contexto": sum(len(f["texto"]) for f in frag),
            "respuesta": resp[:300],
        })
        time.sleep(2)

tabla_k = pd.DataFrame(filas_k)
tabla_k